## 1. Imports & backend

minml's Python bindings live in `_minml`. Build them with

```bash
cmake -B build_py -DMINML_BUILD_PYTHON=ON && cmake --build build_py -j
```

and either add `build_py/` to `PYTHONPATH` or, as below, prepend it inside
the notebook so the kernel doesn't need to know about it.

In [2]:
import sys
sys.path.insert(0, "/home/mbc/minml/build")

import _minml as m
m.set_default_device(m.Device.CUDA)

print("default device:", m.default_device())

default device: Device.CUDA


## 2. Inspecting Tensors

Helper that prints a tensor's shape, dtype, and contents.

In [3]:
def inspect(t):
    is_int = (t.dtype() == m.DType.Int32)
    return {
        "shape": list(t.shape()),
        "dtype": "int32" if is_int else "float32",
        "data": t.tolist_int() if is_int else t.tolist(),
    }

a = m.array([1.0, 2.0, 3.0, 4.0])
b = m.array([10.0, 20.0, 30.0, 40.0])
twos = m.array([2.0, 2.0, 2.0, 2.0])
c = m.mul(m.add(a, b), twos)
inspect(c)

{'shape': [4], 'dtype': 'float32', 'data': [22.0, 44.0, 66.0, 88.0]}

# Simple generative model

Same shape as the Deno notebook: a `Trace` class with a static `simulate`
that draws cluster mixture weights from a Dirichlet, a per-observation
cluster assignment via `randint`, and one observation per assignment via
a Categorical over the assigned cluster's row of `dists`.

`m.vmap` is the curried wrapper around `m.vmap_apply`; the loop, slicing,
and stacking all live in `src/transforms.cpp` and are shared with the
Deno binding.

In [9]:
class Trace:
    obs = None           # (N_obs,) Int32
    assignments = None   # (N_obs,) Int32
    dists = None         # (N_clusters, N_categories) Float32

    @staticmethod
    def simulate(key, N_clusters, N_categories, N_obs):
        t = Trace()
        k1, k2, k3 = key.split(3)
        t.dists = m.Dirichlet(m.ones([N_categories])).sample(k1, [N_clusters])
        t.assignments = m.randint(k2.k0(), k2.k1(), 0, N_clusters,
                                  [N_obs], m.Device.CPU)

        def simulate_one_observation(key, idx):
            return m.Categorical(m.gather(t.dists, idx)).sample(key, [])

        t.obs = m.vmap(simulate_one_observation, [0, 0])(
            k3.split(N_obs), t.assignments)
        return t

Single trace simulation:

In [10]:
trace = Trace.simulate(m.PRNGKey.new(0), 3, 4, 8)
{
    "dists":       inspect(trace.dists),
    "assignments": inspect(trace.assignments),
    "obs":         inspect(trace.obs),
}

{'dists': {'shape': [3, 4],
  'dtype': 'float32',
  'data': [0.026025988161563873,
   0.23222918808460236,
   0.6096919775009155,
   0.13205288350582123,
   0.23686616122722626,
   0.4914940297603607,
   0.09951993077993393,
   0.1721198409795761,
   0.43649742007255554,
   0.25829777121543884,
   0.052322421222925186,
   0.25288236141204834]},
 'assignments': {'shape': [8],
  'dtype': 'int32',
  'data': [2, 0, 0, 2, 1, 1, 1, 0]},
 'obs': {'shape': [8], 'dtype': 'int32', 'data': [0, 2, 2, 3, 1, 1, 0, 2]}}

vmap'd batch of traces. Split the root key first so each trace gets its own:

In [6]:
N_traces = 5
root = m.PRNGKey.new(0)
trace_keys = root.split(N_traces)

traces = m.vmap(Trace.simulate, [0, None, None, None])(
    trace_keys, 3, 4, 10)

{
    "dists":       inspect(traces.dists),
    "assignments": inspect(traces.assignments),
    "obs":         inspect(traces.obs),
}

{'dists': {'shape': [5, 3, 4],
  'dtype': 'float32',
  'data': [0.03578299656510353,
   0.01997232809662819,
   0.4661552309989929,
   0.47808942198753357,
   0.10690134763717651,
   0.2344864308834076,
   0.29967057704925537,
   0.35894161462783813,
   0.022099077701568604,
   0.7258784770965576,
   0.05179296061396599,
   0.2002294957637787,
   0.6547591090202332,
   0.10178259760141373,
   0.16976070404052734,
   0.07369756698608398,
   0.5657760500907898,
   0.03907905891537666,
   0.3266555070877075,
   0.06848937273025513,
   0.08520522713661194,
   0.5631458163261414,
   0.10421077162027359,
   0.2474382221698761,
   0.6410667896270752,
   0.12879157066345215,
   0.2184828668832779,
   0.011658819392323494,
   0.04438493028283119,
   0.1823054701089859,
   0.49737972021102905,
   0.27592992782592773,
   0.03564990311861038,
   0.1475953608751297,
   0.05938933044672012,
   0.757365345954895,
   0.3106776177883148,
   0.10052873194217682,
   0.5603675246238708,
   0.0284261554479